# Ablation Study of 2.5D SwinUNet and Adaptive Attention

### Reproducing the Ablation Study

The experiments were conducted using five random seeds (41, 42, 43, 44, and 45), which were defined in the notebook. Each seed was used to evaluate all five model configurations. The results were then combined for the final analysis.

Cell 1 — Configuration and Setup

In [1]:
# Cell 1 — Configuration and Setup

import os
import random
from pathlib import Path

import gc
import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. Reproducibility
# ============================================================

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_seed(seed: int, deterministic: bool = True) -> None:
    """Set random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        torch.use_deterministic_algorithms(False)


# Seed used for the current experimental run.
# Run the experiments separately with seeds 41, 42, 43, 44, and 45.
SEED = 45
set_seed(SEED, deterministic=False)


# ============================================================
# 2. Device Configuration
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 3. Dataset Paths
# ============================================================

# Set this environment variable to the local dataset directory.
# If it is not set, the default path below will be used.
DATA_ROOT = Path(
    os.environ.get(
        "LIDC_DATA_ROOT",
        r"data\DatasetResized320png"
    )
)

IMAGE_DIR = DATA_ROOT / "resized_images320"
MASK_DIR = DATA_ROOT / "resized_masks320"
METADATA_PATH = DATA_ROOT / "resized_metadata320.csv"

assert IMAGE_DIR.is_dir(), f"Image directory not found: {IMAGE_DIR}"
assert MASK_DIR.is_dir(), f"Mask directory not found: {MASK_DIR}"
assert METADATA_PATH.is_file(), f"Metadata file not found: {METADATA_PATH}"

meta_df = pd.read_csv(METADATA_PATH)

print(f"Metadata rows: {len(meta_df):,}")
print("Dataset paths verified.")


# ============================================================
# 4. Experiment Configuration
# ============================================================

IMG_FORMAT = "PNG"
NUM_WORKERS = 0

PATCH_SIZE = 320
BATCH_SIZE = 16
EPOCHS = 100

IN_CHANNELS = 3 # Number of input channels (3 adjacent CT slices for 2.5D input; set to 1 for the 2D baseline)
DATA_PERCENT = 1.0


Device: cuda
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Metadata rows: 38,229
Dataset paths verified.


Cell 2 — Data Filtering and Patient-Level Splitting

In [2]:
# Cell 2 — Data Filtering and Patient-Level Splitting

import re

import numpy as np
from sklearn.model_selection import GroupShuffleSplit


# ============================================================
# 1. Filter Segmentation Samples
# ============================================================

# Fixed random state for patient-level dataset splitting.
# This split remains unchanged across model-training seeds.
dataSeed = 42

df = meta_df.copy()

# Exclude samples with malignancy score 3 from the segmentation dataset.
# These samples may be used separately for classification, according
# to the selection criteria defined for that task.
df_filtered = df[df["malignancy"] != 3].reset_index(drop=True)

# Segmentation-specific filtering: retain samples with a polygon area
# of at least 100 pixels. This criterion is not applied to the
# classification dataset, which uses separate sample-selection criteria.
df_new = df_filtered[df_filtered["polygon_area"] >= 100].copy()

removed = len(df) - len(df_new)
removed_pct = removed / len(df) * 100 if len(df) > 0 else 0.0

print("Segmentation dataset filtering")
print(f"Total metadata rows: {len(df):,}")
print(f"Rows after malignancy filtering: {len(df_filtered):,}")
print(f"Rows after polygon-area filtering: {len(df_new):,}")
print(f"Total rows removed: {removed:,} ({removed_pct:.2f}%)")

# Create a set of valid image filenames.
good_images = {
    f"{name}.png" for name in df_new["image_name"].astype(str)
}

print(f"Valid image filenames: {len(good_images):,}")
print("-" * 50)


# ============================================================
# 2. Match Images with Masks and Patient IDs
# ============================================================

imgs = sorted(IMAGE_DIR.glob("*.png"))
assert imgs, f"No PNG images found in {IMAGE_DIR}"


def parse_id_idx(path: Path):
    """Extract the patient identifier and slice index from a filename."""
    match = re.search(
        r"(LIDC.*?\d+|patient[_-]?\d+).*?(\d+)\.png$",
        path.name,
        re.IGNORECASE,
    )

    if match:
        patient_id = match.group(1)
        slice_idx = int(match.group(2))
    else:
        patient_id = path.stem.split("_")[0]
        slice_idx = 0

    return patient_id, slice_idx


pairs = []

for image_path in imgs:
    if image_path.name not in good_images:
        continue

    mask_path = MASK_DIR / image_path.name

    if mask_path.is_file():
        patient_id, slice_idx = parse_id_idx(image_path)
        pairs.append((image_path, mask_path, patient_id, slice_idx))

# Sort samples for consistent ordering.
pairs.sort(key=lambda item: (item[2], item[3]))

assert pairs, "No matching image-mask pairs found after filtering."

print(f"Matched image-mask pairs: {len(pairs):,}")
print("-" * 50)


# ============================================================
# 3. Patient-Level Train/Validation/Test Split
# ============================================================

indices = np.arange(len(pairs))
groups = np.array([item[2] for item in pairs])

# First split: 70% training and 30% held out for validation and testing.
gss1 = GroupShuffleSplit(
    test_size=0.30,
    random_state=dataSeed,
)
train_idx, val_test_idx = next(
    gss1.split(indices, groups=groups)
)

# Second split: divide the held-out 30% equally between validation and test.
held_out_groups = groups[val_test_idx]

gss2 = GroupShuffleSplit(
    test_size=0.50,
    random_state=dataSeed,
)
val_rel_idx, test_rel_idx = next(
    gss2.split(val_test_idx, groups=held_out_groups)
)

train_ids = indices[train_idx]
val_ids = val_test_idx[val_rel_idx]
test_ids = val_test_idx[test_rel_idx]

print("Dataset split")
print(f"Total image-mask pairs: {len(indices):,}")
print(f"Training samples: {len(train_ids):,}")
print(f"Validation samples: {len(val_ids):,}")
print(f"Test samples: {len(test_ids):,}")
print("-" * 50)


# ============================================================
# 4. Check Patient-Level Separation
# ============================================================

train_patients = set(groups[train_ids])
val_patients = set(groups[val_ids])
test_patients = set(groups[test_ids])

print("Patient overlap checks")
print("Train–validation:", train_patients & val_patients)
print("Train–test:", train_patients & test_patients)
print("Validation–test:", val_patients & test_patients)

print("\nNumber of patients per split")
print(f"Training: {len(train_patients):,}")
print(f"Validation: {len(val_patients):,}")
print(f"Test: {len(test_patients):,}")


Segmentation dataset filtering
Total metadata rows: 38,229
Rows after malignancy filtering: 26,431
Rows after polygon-area filtering: 13,943
Total rows removed: 24,286 (63.53%)
Valid image filenames: 13,943
--------------------------------------------------
Matched image-mask pairs: 13,943
--------------------------------------------------
Dataset split
Total image-mask pairs: 13,943
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
--------------------------------------------------
Patient overlap checks
Train–validation: set()
Train–test: set()
Validation–test: set()

Number of patients per split
Training: 366
Validation: 79
Test: 79


Cell 3 — Dataset, Preprocessing, and DataLoaders

In [3]:
# Cell 3 — Dataset, Preprocessing, and DataLoaders
import random
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
from scipy.ndimage import label as cc_label
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


# ============================================================
# 1. Image Loading
# ============================================================

def load_img(path: Path) -> np.ndarray:
    """Load a grayscale PNG image and normalize pixel values to [0, 1]."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")

    return img.astype(np.float32) / 255.0


# ============================================================
# 2. 2D and 2.5D Input Construction
# ============================================================

def stack_25d(idx_in_all: int, k: int = 3) -> np.ndarray:
    """
    Construct a multi-channel input from adjacent CT slices.

    For k=3, use the previous, central, and next slices.
    For k=1, use only the central slice.
    """
    _, _, patient_id, _ = pairs[idx_in_all]

    half = k // 2
    offsets = range(-half, half + 1) if k > 1 else [0]
    channels = []

    for offset in offsets:
        j = idx_in_all + offset

        # Use the central slice when the index is out of bounds.
        if j < 0 or j >= len(pairs):
            j = idx_in_all

        _, _, neighbor_patient_id, _ = pairs[j]

        # Prevent stacking slices from different patients.
        if neighbor_patient_id != patient_id:
            j = idx_in_all

        channels.append(load_img(pairs[j][0]))

    return np.stack(channels, axis=0)


# ============================================================
# 3. ROI Localization for Patch-Based Analysis
# ============================================================

def largest_blob_center(
    mask: np.ndarray,
    min_area: int = 20,
):
    """Return the centroid of the largest connected component."""
    binary_mask = (mask > 0.5).astype(np.uint8)

    if binary_mask.sum() < min_area:
        return None

    labeled_mask, num_components = cc_label(binary_mask)

    if num_components == 0:
        return None

    largest_area = 0
    center_y, center_x = None, None

    for component_id in range(1, num_components + 1):
        ys, xs = np.where(labeled_mask == component_id)
        area = len(ys)

        if area > largest_area:
            largest_area = area
            center_y = int(ys.mean())
            center_x = int(xs.mean())

    return center_y, center_x


def get_crop_coords(
    mask: np.ndarray,
    patch_size: int,
    crop_mode: str = "random",
):
    """
    Return crop coordinates for ROI-based or random patch extraction.

    ROI cropping is intended only for the patch-based analysis.
    Random cropping is available for experiments that explicitly
    enable cropping. Full-size experiments must use use_crop=False.
    """
    height, width = mask.shape

    if patch_size > height or patch_size > width:
        raise ValueError(
            f"Patch size {patch_size} exceeds image dimensions "
            f"({height}, {width})."
        )

    if crop_mode == "roi":
        center = largest_blob_center(mask)

        if center is None:
            center_y, center_x = height // 2, width // 2
        else:
            center_y, center_x = center

        half = patch_size // 2
        y1 = np.clip(center_y - half, 0, height - patch_size)
        x1 = np.clip(center_x - half, 0, width - patch_size)

        return int(y1), int(x1)

    elif crop_mode == "random":
        y1 = random.randint(0, height - patch_size)
        x1 = random.randint(0, width - patch_size)

        return y1, x1

    else:
        raise ValueError(
            f"Unknown crop mode: {crop_mode}. "
            "Supported modes are 'roi' and 'random'."
        )


# ============================================================
# 4. Optional Sample Filtering
# ============================================================

def filter_indices(
    indices,
    fraction: float = 1.0,
    max_slices_per_patient=None,
):
    """
    Optionally select a fraction of patients and limit the number
    of slices per patient.

    By default, all supplied indices are retained.
    """
    if fraction == 1.0 and max_slices_per_patient is None:
        return list(indices)

    patient_to_indices = defaultdict(list)

    for idx in indices:
        _, _, patient_id, _ = pairs[idx]
        patient_to_indices[patient_id].append(idx)

    patient_ids = list(patient_to_indices.keys())

    if fraction < 1.0:
        random.shuffle(patient_ids)
        num_patients = max(1, int(len(patient_ids) * fraction))
        patient_ids = patient_ids[:num_patients]

    filtered_indices = []

    for patient_id in patient_ids:
        patient_indices = patient_to_indices[patient_id].copy()

        if (
            max_slices_per_patient is not None
            and len(patient_indices) > max_slices_per_patient
        ):
            random.shuffle(patient_indices)
            patient_indices = patient_indices[:max_slices_per_patient]

        filtered_indices.extend(patient_indices)

    return filtered_indices


# ============================================================
# 5. Segmentation Dataset
# ============================================================

class AblationDataset(Dataset):
    """
    Dataset for segmentation ablation experiments.

    Full-size inputs are used when use_crop=False.
    ROI and random cropping are applied only when use_crop=True.
    """

    def __init__(
        self,
        id_list,
        train: bool = True,
        use_25d: bool = True,
        crop_mode: str = "random",
        use_crop: bool = False,
    ):
        self.ids = list(id_list)
        self.train = train
        self.use_25d = use_25d
        self.crop_mode = crop_mode
        self.use_crop = use_crop

        if self.crop_mode not in {"roi", "random"}:
            raise ValueError(
                "crop_mode must be either 'roi' or 'random'."
            )

        # Three channels for 2.5D input; one for 2D input.
        self.k = 3 if use_25d else 1

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        idx = self.ids[i]

        # Load the input image and its corresponding segmentation mask.
        x = stack_25d(idx, k=self.k)
        y = load_img(pairs[idx][1])

        # ROI or random cropping is used only when explicitly enabled.
        if self.use_crop:
            y1, x1 = get_crop_coords(
                y,
                PATCH_SIZE,
                crop_mode=self.crop_mode,
            )

            x = x[
                :,
                y1:y1 + PATCH_SIZE,
                x1:x1 + PATCH_SIZE,
            ]
            y = y[
                y1:y1 + PATCH_SIZE,
                x1:x1 + PATCH_SIZE,
            ]

        # Apply random flips during training only.
        if self.train:
            if random.random() < 0.5:
                x = x[:, :, ::-1]
                y = y[:, ::-1]

            if random.random() < 0.5:
                x = x[:, ::-1, :]
                y = y[::-1, :]

        # Ensure arrays have compatible memory layouts for PyTorch.
        x = np.ascontiguousarray(x)
        y = np.ascontiguousarray(y)

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float().unsqueeze(0),
        )


# ============================================================
# 6. DataLoader Construction
# ============================================================

def make_loaders_ablation(
    train_ids,
    val_ids,
    test_ids,
    use_25d: bool = True,
    crop_mode: str = "random",
    use_crop: bool = False,
    batch_size: int = BATCH_SIZE,
    fraction: float = 1.0,
    max_slices_per_patient=None,
    num_workers: int = NUM_WORKERS,
):
    """
    Create training, validation, and test DataLoaders.

    For full-size experiments, set use_crop=False.
    In this case, crop_mode does not affect the input images.
    """

    # Optionally reduce the number of samples.
    train_filtered = filter_indices(
        train_ids, fraction, max_slices_per_patient
    )
    val_filtered = filter_indices(
        val_ids, fraction, max_slices_per_patient
    )
    test_filtered = filter_indices(
        test_ids, fraction, max_slices_per_patient
    )

    print(f"Training samples: {len(train_filtered):,}")
    print(f"Validation samples: {len(val_filtered):,}")
    print(f"Test samples: {len(test_filtered):,}")

    # Assign higher sampling weights to slices containing a foreground mask.
    sample_weights = []

    for idx in train_filtered:
        mask = load_img(pairs[idx][1])
        weight = 6.0 if (mask > 0.5).any() else 1.0
        sample_weights.append(weight)

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_filtered),
        replacement=True,
    )

    # Training uses the requested cropping configuration.
    train_dataset = AblationDataset(
        train_filtered,
        train=True,
        use_25d=use_25d,
        crop_mode=crop_mode,
        use_crop=use_crop,
    )

    # Validation and test use full-size images.
    val_dataset = AblationDataset(
        val_filtered,
        train=False,
        use_25d=use_25d,
        crop_mode="random",
        use_crop=False,
    )

    test_dataset = AblationDataset(
        test_filtered,
        train=False,
        use_25d=use_25d,
        crop_mode="random",
        use_crop=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader


Cell 4 — Loss Function (Hybrid Loss)

In [4]:
# Cell 4 — Loss Function (Hybrid Loss)

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. Focal Tversky Loss
# ============================================================

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=1.33, eps=1e-6):
        super().__init__()
        self.a = alpha
        self.b = beta
        self.g = gamma
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)

        tp = (p * targets).sum(dim=(2, 3))
        fp = (p * (1 - targets)).sum(dim=(2, 3))
        fn = ((1 - p) * targets).sum(dim=(2, 3))

        t = (tp + self.eps) / (
            tp + self.a * fp + self.b * fn + self.eps
        )

        return (1.0 - t).clamp_min(1e-8).pow(self.g).mean()


# ============================================================
# 2. Soft Dice Loss
# ============================================================

class SoftDiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)

        num = 2 * (p * targets).sum(dim=(2, 3))
        den = (p + targets).sum(dim=(2, 3)) + self.eps

        return (1.0 - (num / den)).mean()


# ============================================================
# 3. Boundary Loss
# ============================================================

class BoundaryLoss(nn.Module):
    def __init__(self):
        super().__init__()

        # Sobel kernels for horizontal and vertical gradients.
        kx = torch.tensor(
            [[[-1, 0, 1],
              [-2, 0, 2],
              [-1, 0, 1]]],
            dtype=torch.float32,
        ).view(1, 1, 3, 3)

        ky = torch.tensor(
            [[[1, 2, 1],
              [0, 0, 0],
              [-1, -2, -1]]],
            dtype=torch.float32,
        ).view(1, 1, 3, 3)

        # Register kernels as buffers so they follow the module's device.
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)

        gx_p = F.conv2d(p, self.kx, padding=1)
        gy_p = F.conv2d(p, self.ky, padding=1)

        gx_t = F.conv2d(targets, self.kx, padding=1)
        gy_t = F.conv2d(targets, self.ky, padding=1)

        bp = torch.sqrt(gx_p**2 + gy_p**2 + 1e-6)
        bt = torch.sqrt(gx_t**2 + gy_t**2 + 1e-6)

        return F.l1_loss(bp, bt)


# ============================================================
# 4. Combined Dice-Tversky Loss
# ============================================================

class ComboDiceTverskyLoss(nn.Module):
    def __init__(
        self,
        w_ft=0.25,
        w_dice=0.25,
        w_bce=0.35,
        w_bd=0.15,
        pos_weight=4.0,
    ):
        super().__init__()

        self.ft = FocalTverskyLoss()
        self.dice = SoftDiceLoss()
        self.bd = BoundaryLoss()

        # The four main loss weights sum to 1.0.
        self.weights = [w_ft, w_dice, w_bce, w_bd]

        # Positive-class weight for binary cross-entropy.
        self.register_buffer("posw", torch.tensor(pos_weight))

    def forward(
        self,
        logits,
        targets,
        aux2=None,
        aux3=None,
        attn_mask=None,
    ):
        # Resize the target mask if its spatial dimensions differ
        # from those of the main prediction.
        targets_m = (
            targets
            if logits.shape[-2:] == targets.shape[-2:]
            else F.interpolate(
                targets,
                size=logits.shape[-2:],
                mode="nearest",
            )
        )

        # Main segmentation loss.
        loss = (
            self.weights[0] * self.ft(logits, targets_m)
            + self.weights[1] * self.dice(logits, targets_m)
            + self.weights[2]
            * F.binary_cross_entropy_with_logits(
                logits,
                targets_m,
                pos_weight=self.posw,
            )
            + self.weights[3] * self.bd(logits, targets_m)
        )

        # Auxiliary supervision at the second decoder output.
        if aux2 is not None:
            t2 = F.interpolate(
                targets,
                size=aux2.shape[-2:],
                mode="nearest",
            )
            loss += 0.15 * F.binary_cross_entropy_with_logits(
                aux2,
                t2,
                pos_weight=self.posw,
            )

        # Auxiliary supervision at the third decoder output.
        if aux3 is not None:
            t3 = F.interpolate(
                targets,
                size=aux3.shape[-2:],
                mode="nearest",
            )
            loss += 0.1 * F.binary_cross_entropy_with_logits(
                aux3,
                t3,
                pos_weight=self.posw,
            )

        # Optional attention-map supervision using mean squared error.
        if attn_mask is not None:
            t_attn = F.interpolate(
                targets,
                size=attn_mask.shape[-2:],
                mode="nearest",
            )
            loss += 0.2 * F.mse_loss(attn_mask, t_attn)

        return loss


# Initialize the combined loss function.
loss_fn = ComboDiceTverskyLoss()


Cell 5 — SwinUNet Model 

In [6]:
# Cell 5 — SwinUNet Model 

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
# GroupNorm
# ============================================================

def GN(ch, groups=8):
    g = math.gcd(groups, ch)
    return nn.GroupNorm(g, ch)


# ============================================================
# Spatial Attention
# ============================================================

class SpatialAttention(nn.Module):
    """
    Spatial Attention (CBAM style)

    Input:
        (B,C,H,W)

    Output:
        mask : (B,1,H,W)
    """

    def __init__(self, kernel_size=7):
        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

        # ---------- Explainability ----------
        self.last_avg = None
        self.last_max = None
        self.last_concat = None
        self.last_logits = None
        self.last_mask = None

    def forward(self, x):

        avg_out = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x_cat = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        logits = self.conv(x_cat)

        mask = self.sigmoid(logits)

        # ==================================================
        # Save everything for visualization / XAI
        # ==================================================

        self.last_avg = avg_out.detach()

        self.last_max = max_out.detach()

        self.last_concat = x_cat.detach()

        self.last_logits = logits.detach()

        self.last_mask = mask.detach()

        return mask


# ============================================================
# Adaptive Attention
# ============================================================

class AdaptiveAttention(nn.Module):

    def __init__(
        self,
        c,
        r=16,
        use_residual=True,
        combine_method="max"
    ):

        super().__init__()

        self.use_residual = use_residual

        self.combine_method = combine_method

        self.gn = GN(c)

        mid_ch = max(1, c // r)

        # ------------------------------------
        # Spatial
        # ------------------------------------

        self.spatial_att = SpatialAttention(
            kernel_size=7
        )
    
        # ------------------------------------
        # Local Attention
        # ------------------------------------

        self.local_attn = nn.Sequential(

            nn.Conv2d(
                c,
                mid_ch,
                kernel_size=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                mid_ch,
                c,
                kernel_size=3,
                padding=1
            ),

            nn.Sigmoid()
        )

        # ------------------------------------
        # Global Attention
        # ------------------------------------

        self.global_attn = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                c,
                mid_ch,
                kernel_size=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                mid_ch,
                c,
                kernel_size=1
            ),

            nn.Sigmoid()

        )

        if self.use_residual:

            self.res_scale = nn.Parameter(
                torch.ones(1) * 0.1
            )

        # ===================================================
        # Explainability Storage
        # ===================================================

        self.cache = {}

        self.activations = None

        self.gradients = None

    # =======================================================
    # Gradient Hook
    # =======================================================

    def save_gradient(self, grad):

        self.gradients = grad.detach()

    # =======================================================
    # Forward
    # =======================================================

    def forward(self, x):

        # -----------------------------
        # input
        # -----------------------------

        self.cache.clear()

        self.cache["input"] = x.detach()

        # -----------------------------
        # normalization
        # -----------------------------

        x_norm = self.gn(x)

        self.cache["normalized"] = x_norm.detach()

        # -----------------------------
        # spatial attention
        # -----------------------------

        spatial_mask = self.spatial_att(x_norm)

        self.cache["spatial_mask"] = spatial_mask.detach()

        x_focused = x_norm * spatial_mask

        self.cache["focused"] = x_focused.detach()

        # -----------------------------
        # local attention
        # -----------------------------

        local_attn = self.local_attn(x_focused)

        self.cache["local_attention"] = local_attn.detach()

        # -----------------------------
        # global attention
        # -----------------------------

        global_attn = self.global_attn(x_focused)

        self.cache["global_attention"] = global_attn.detach()

        # -----------------------------
        # combine
        # -----------------------------

        if self.combine_method == "max":

            combined_attn = torch.max(
                local_attn,
                global_attn
            )

        elif self.combine_method == "avg":

            combined_attn = (
                local_attn + global_attn
            ) / 2

        elif self.combine_method == "sum":

            combined_attn = torch.clamp(
                local_attn + global_attn,
                0,
                1
            )

        else:

            combined_attn = torch.max(
                local_attn,
                global_attn
            )

        self.cache["combined_attention"] = combined_attn.detach()

        # -----------------------------
        # final attention
        # -----------------------------

        final_attn = combined_attn * spatial_mask

        self.cache["final_attention"] = final_attn.detach()

        # -----------------------------
        # weighted feature
        # -----------------------------

        weighted = x * final_attn

        self.cache["weighted_feature"] = weighted.detach()

        # -----------------------------
        # residual
        # -----------------------------

        if self.use_residual:

            out = weighted + self.res_scale * x

        else:

            out = weighted

        self.cache["residual_output"] = out.detach()

        # -----------------------------
        # final output
        # -----------------------------

        final_out = out * spatial_mask

        self.cache["output"] = final_out.detach()

        # -----------------------------
        # Activations
        # -----------------------------

        self.activations = final_out

        if final_out.requires_grad:

            final_out.register_hook(
                self.save_gradient
            )

        return final_out, spatial_mask
    
    # ============================================================
# Decoder Block
# ============================================================

class Up2(nn.Module):

    def __init__(self, in_ch, skip_ch, out_ch):

        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_ch,
            in_ch // 2,
            kernel_size=2,
            stride=2
        )

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_ch // 2 + skip_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),

            nn.ReLU(inplace=True)

        )

        # ====================================
        # Explainability
        # ====================================

        self.cache = {}

        self.activations = None

        self.gradients = None

    def save_gradient(self, grad):

        self.gradients = grad.detach()

    def forward(self, x, skip):

        self.cache.clear()

        self.cache["decoder_input"] = x.detach()

        self.cache["skip"] = skip.detach()

        x = self.up(x)

        self.cache["upsampled"] = x.detach()

        if x.shape[-2:] != skip.shape[-2:]:

            x = F.interpolate(

                x,

                size=skip.shape[-2:],

                mode="bilinear",

                align_corners=False

            )

        self.cache["aligned"] = x.detach()

        merged = torch.cat([skip, x], dim=1)

        self.cache["merged"] = merged.detach()

        out = self.conv(merged)

        self.cache["output"] = out.detach()

        self.activations = out

        if out.requires_grad:

            out.register_hook(self.save_gradient)

        return out


# ============================================================
# Tiny CNN Encoder
# ============================================================

class TinySwinLike(nn.Module):

    def __init__(self, in_chans):

        super().__init__()

        def block(i, o):

            return nn.Sequential(

                nn.Conv2d(

                    i,

                    o,

                    kernel_size=3,

                    padding=1,

                    bias=False

                ),

                GN(o),

                nn.ReLU(inplace=True),

                nn.Conv2d(

                    o,

                    o,

                    kernel_size=3,

                    padding=1,

                    bias=False

                ),

                GN(o),

                nn.ReLU(inplace=True)

            )

        self.s1 = block(in_chans, 64)

        self.s2 = block(64, 128)

        self.s3 = block(128, 256)

        self.s4 = block(256, 512)

        self.pool = nn.MaxPool2d(2)

        # ====================================
        # Explainability
        # ====================================

        self.cache = {}

    def forward(self, x):

        self.cache.clear()

        self.cache["input"] = x.detach()

        f1 = self.s1(x)

        self.cache["stage1"] = f1.detach()

        p1 = self.pool(f1)

        self.cache["pool1"] = p1.detach()

        f2 = self.s2(p1)

        self.cache["stage2"] = f2.detach()

        p2 = self.pool(f2)

        self.cache["pool2"] = p2.detach()

        f3 = self.s3(p2)

        self.cache["stage3"] = f3.detach()

        p3 = self.pool(f3)

        self.cache["pool3"] = p3.detach()

        f4 = self.s4(p3)

        self.cache["stage4"] = f4.detach()

        return [

            f1,

            f2,

            f3,

            f4

        ]


# ============================================================
# Explainability Helper
# ============================================================

class ExplainContainer:

    """
    Stores intermediate information used for:

    Attention Visualization
    Grad-CAM
    Occlusion Sensitivity
    SHAP

    """

    def __init__(self):

        self.encoder = {}

        self.decoder = {}

        self.attention = {}

        self.outputs = {}

        self.meta = {}

    def clear(self):

        self.encoder.clear()

        self.decoder.clear()

        self.attention.clear()

        self.outputs.clear()

        self.meta.clear()

        # ============================================================
# Main Model
# ============================================================

class SwinUNetAdaptive_DS(nn.Module):

    def __init__(
        self,
        in_ch=IN_CHANNELS,
        base="timm",
        out_ch=1,
        use_enc_att=True,
        use_dec_att=True,
        use_low_att=False,
        att_combine_method="max",
        patch_size=PATCH_SIZE,
    ):

        super().__init__()

        self.use_enc_att = use_enc_att
        self.use_dec_att = use_dec_att
        self.use_low_att = use_low_att

        self.uses_timm = (base == "timm")

        self.patch_size = patch_size

        # ====================================================
        # Explainability Container
        # ====================================================

        self.explain = ExplainContainer()

        self.activations = {}
        self.gradients = {}

        # ====================================================
        # Encoder
        # ====================================================

        if self.uses_timm:

            import timm

            self.encoder = timm.create_model(
                "swin_tiny_patch4_window7_224",
                pretrained=True,
                features_only=True,
                out_indices=(0, 1, 2, 3),
                in_chans=in_ch,
                img_size=self.patch_size,
            )

            self.feat_chs = [96, 192, 384, 768]

        else:

            self.encoder = TinySwinLike(in_ch)

            self.feat_chs = [64, 128, 256, 512]

        c1, c2, c3, c4 = self.feat_chs

        # ====================================================
        # Encoder Attention
        # ====================================================

        self.enc_att2 = (
            AdaptiveAttention(
                c2,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_low_att
            else nn.Identity()
        )

        self.enc_att3 = (
            AdaptiveAttention(
                c3,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_enc_att
            else nn.Identity()
        )

        self.enc_att4 = (
            AdaptiveAttention(
                c4,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_enc_att
            else nn.Identity()
        )

        # ====================================================
        # Decoder
        # ====================================================

        self.up1 = Up2(c4, c3, c3)

        self.dec_att1 = (
            AdaptiveAttention(
                c3,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_dec_att
            else nn.Identity()
        )

        self.up2 = Up2(c3, c2, c2)

        self.dec_att2 = (
            AdaptiveAttention(
                c2,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_dec_att
            else nn.Identity()
        )

        self.up3 = Up2(c2, c1, c1)

        self.dec_att3 = (
            AdaptiveAttention(
                c1,
                use_residual=True,
                combine_method=att_combine_method,
            )
            if use_dec_att
            else nn.Identity()
        )

        # ====================================================
        # Prediction Heads
        # ====================================================

        self.out = nn.Conv2d(
            c1,
            out_ch,
            kernel_size=1
        )

        self.aux2 = nn.Conv2d(
            c2,
            out_ch,
            kernel_size=1
        )

        self.aux3 = nn.Conv2d(
            c3,
            out_ch,
            kernel_size=1
        )

        # ====================================================
        # Weight Initialization
        # ====================================================

        self._initialize_weights()

    # ========================================================
    # Weight Initialization
    # ========================================================

    def _initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):

                nn.init.kaiming_normal_(
                    m.weight,
                    mode="fan_out",
                    nonlinearity="relu"
                )

                if m.bias is not None:

                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):

                nn.init.constant_(m.weight, 1)

                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.GroupNorm):

                nn.init.constant_(m.weight, 1)

                nn.init.constant_(m.bias, 0)

        # ---------------------------------------------
        # Residual scale initialization
        # ---------------------------------------------

        for module in self.modules():

            if isinstance(module, AdaptiveAttention):

                if hasattr(module, "res_scale"):

                    with torch.no_grad():

                        module.res_scale.fill_(0.1)

    # ========================================================
    # Gradient Hook (GradCAM و Captum)
    # ========================================================

    def _save_gradient(self, name):

        def hook(grad):

            self.gradients[name] = grad.detach()

        return hook
    
        # ========================================================
    # Forward
    # ========================================================

    def forward(self, x):

        self.explain.clear()

        B, C, H, W = x.shape

        original_size = (H, W)

        self.explain.meta["input_size"] = original_size

        self.explain.meta["batch_size"] = B

        self.explain.meta["channels"] = C

        # ====================================================
        # Encoder
        # ====================================================

        if self.uses_timm:

            self.explain.meta["input"] = x.detach()

            x_big = F.interpolate(
                x,
                (self.patch_size, self.patch_size),
                mode="bilinear",
                align_corners=False
            )

            self.explain.meta["encoder_input"] = x_big.detach()

            feats = self.encoder(x_big)

            feats = [
                f.permute(0, 3, 1, 2).contiguous()
                for f in feats
            ]

        else:

            feats = self.encoder(x)

        f1, f2, f3, f4 = feats

        # ----------------------------------------------------
        # Save raw encoder features
        # ----------------------------------------------------

        self.explain.encoder["stage1"] = f1.detach()
        self.explain.encoder["stage2"] = f2.detach()
        self.explain.encoder["stage3"] = f3.detach()
        self.explain.encoder["stage4"] = f4.detach()

        # ====================================================
        # Encoder Attention
        # ====================================================

        if self.use_low_att:

            f2, mask2 = self.enc_att2(f2)

            self.explain.attention["enc2"] = self.enc_att2.cache

            self.explain.attention["enc2_mask"] = mask2.detach()

        if self.use_enc_att:

            f3, mask3 = self.enc_att3(f3)

            self.explain.attention["enc3"] = self.enc_att3.cache

            self.explain.attention["enc3_mask"] = mask3.detach()

            f4, mask4 = self.enc_att4(f4)

            self.explain.attention["enc4"] = self.enc_att4.cache

            self.explain.attention["enc4_mask"] = mask4.detach()

        self.explain.encoder["stage2_after"] = f2.detach()
        self.explain.encoder["stage3_after"] = f3.detach()
        self.explain.encoder["stage4_after"] = f4.detach()

        # ====================================================
        # Decoder 1
        # ====================================================

        y1 = self.up1(f4, f3)

        self.explain.decoder["up1"] = self.up1.cache

        if self.use_dec_att:

            y1, mask_d1 = self.dec_att1(y1)

            self.explain.attention["dec1"] = self.dec_att1.cache

            self.explain.attention["dec1_mask"] = mask_d1.detach()

        self.explain.decoder["dec1_output"] = y1.detach()

        a3 = self.aux3(y1)

        self.explain.outputs["aux3_raw"] = a3.detach()

        # ====================================================
        # Decoder 2
        # ====================================================

        y2 = self.up2(y1, f2)

        self.explain.decoder["up2"] = self.up2.cache

        if self.use_dec_att:

            y2, mask_d2 = self.dec_att2(y2)

            self.explain.attention["dec2"] = self.dec_att2.cache

            self.explain.attention["dec2_mask"] = mask_d2.detach()

        self.explain.decoder["dec2_output"] = y2.detach()

        a2 = self.aux2(y2)

        self.explain.outputs["aux2_raw"] = a2.detach()

        # ====================================================
        # Decoder 3
        # ====================================================

        y3 = self.up3(y2, f1)

        self.explain.decoder["up3"] = self.up3.cache

        mask_d3 = None

        if self.use_dec_att:

            y3, mask_d3 = self.dec_att3(y3)

            self.explain.attention["dec3"] = self.dec_att3.cache

            self.explain.attention["dec3_mask"] = mask_d3.detach()

        self.explain.decoder["dec3_output"] = y3.detach()

        # ====================================================
        # Final Prediction
        # ====================================================

        y = self.out(y3)

        self.explain.outputs["logits_raw"] = y.detach()

        # ====================================================
        # Register Gradients
        # ====================================================

        if y3.requires_grad:

            self.activations["decoder_last"] = y3

            y3.register_hook(
                self._save_gradient("decoder_last")
            )

        if f4.requires_grad:

            self.activations["encoder_last"] = f4

            f4.register_hook(
                self._save_gradient("encoder_last")
            )

        # ====================================================
        # Resize
        # ====================================================

        y = F.interpolate(
            y,
            original_size,
            mode="bilinear",
            align_corners=False
        )

        a2 = F.interpolate(
            a2,
            original_size,
            mode="bilinear",
            align_corners=False
        )

        a3 = F.interpolate(
            a3,
            original_size,
            mode="bilinear",
            align_corners=False
        )

        self.explain.outputs["prediction"] = y.detach()

        self.explain.outputs["aux2"] = a2.detach()

        self.explain.outputs["aux3"] = a3.detach()

        if mask_d3 is not None:

            mask_d3 = F.interpolate(
                mask_d3,
                original_size,
                mode="bilinear",
                align_corners=False
            )

            self.explain.outputs["mask_d3"] = mask_d3.detach()

        # ====================================================
        # Return
        # ====================================================

        if self.training:

            return y, a2, a3, mask_d3

        return y, a2, a3
    

Cell 6 — Training and Evaluation Metrics

In [7]:
# Cell 6 — Training and Evaluation Metrics

import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from scipy.spatial.distance import directed_hausdorff

# ====================================================================
# Threshold Tuning
# ====================================================================
@torch.no_grad()
def find_best_thr(model, loader, device, thr_list=None, max_points=2_000_000):
    model.eval()
    if thr_list is None:
        thr_list = np.linspace(0.1, 0.9, 17)

    probs, gts = [], []

    use_amp = (device.type == "cuda")
    amp_dtype = torch.float16

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp
        ):
            logits, _, _ = model(xb)

        p = torch.sigmoid(logits)
        probs.append(p.detach().cpu().numpy().reshape(-1))
        gts.append(yb.detach().cpu().numpy().reshape(-1))

    probs = np.concatenate(probs) if len(probs) else np.array([], dtype=np.float32)
    gts = np.concatenate(gts) if len(gts) else np.array([], dtype=np.float32)

    if probs.size == 0:
        return 0.5, 0.0

    # Subsample pixels if needed to limit memory usage.
    if probs.size > max_points:
        idx = np.random.choice(probs.size, max_points, replace=False)
        probs, gts = probs[idx], gts[idx]

    best_thr, best_dice = 0.5, 0.0
    eps = 1e-6
    gts = (gts > 0.5).astype(np.float32)

    for t in thr_list:
        pred = (probs > t).astype(np.float32)
        tp = (pred * gts).sum()
        fp = (pred * (1.0 - gts)).sum()
        fn = ((1.0 - pred) * gts).sum()
        dice = (2.0 * tp) / (2.0 * tp + fp + fn + eps)

        if dice > best_dice:
            best_dice, best_thr = float(dice), float(t)

    return best_thr, best_dice


# ====================================================================
# Train
# ====================================================================
def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    n_batches = 0

    amp_enabled = use_amp and (device.type == "cuda")
    amp_dtype = torch.float16

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled
        ):
            main_output, aux2, aux3, mask_d3 = model(xb)
            loss = loss_fn(
                main_output,
                yb,
                aux2=aux2,
                aux3=aux3,
                attn_mask=mask_d3
            )

        if not torch.isfinite(loss):
            print(f"[WARN] Non-finite loss detected: {loss.item()}, skipping batch.")
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        n_batches += 1

    return total_loss / max(n_batches, 1)


# ====================================================================
# Evaluation: Main Metrics, HD95, AUC/AP, and Recall@95% Specificity
# ====================================================================
@torch.no_grad()
def eval_full_split(
    model,
    loader,
    loss_fn,
    device,
    thr=0.5,
    calc_hd=False
):
    model.eval()

    total_loss = 0.0
    n_batches = 0

    probs_list, gts_list = [], []
    TP, FP, FN, TN = 0.0, 0.0, 0.0, 0.0
    hd_list = []

    amp_enabled = (device.type == "cuda")
    amp_dtype = torch.float16

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled
        ):
            # Evaluation returns three outputs.
            main_output, aux2, aux3 = model(xb)
            loss = loss_fn(
                main_output,
                yb,
                aux2=aux2,
                aux3=aux3
            )

        total_loss += float(loss.item())
        n_batches += 1

        p = torch.sigmoid(main_output)
        pred = (p > thr).float()

        probs_list.append(p.detach().cpu().numpy().reshape(-1))
        gts_list.append(yb.detach().cpu().numpy().reshape(-1))

        TP += (pred * yb).sum().item()
        FP += (pred * (1 - yb)).sum().item()
        FN += ((1 - pred) * yb).sum().item()
        TN += ((1 - pred) * (1 - yb)).sum().item()

        # ---------- HD95 ----------
        if calc_hd:
            pred_b = pred.detach().cpu().numpy().astype(bool)
            gt_b = yb.detach().cpu().numpy().astype(bool)

            for i in range(pred_b.shape[0]):
                p_mask = pred_b[i, 0] if pred_b.ndim == 4 else pred_b[i]
                g_mask = gt_b[i, 0] if gt_b.ndim == 4 else gt_b[i]

                if np.sum(g_mask) > 0 and np.sum(p_mask) > 0:
                    pred_coords = np.argwhere(p_mask)
                    gt_coords = np.argwhere(g_mask)

                    d1 = directed_hausdorff(pred_coords, gt_coords)[0]
                    d2 = directed_hausdorff(gt_coords, pred_coords)[0]
                    hd_list.append(max(d1, d2))

    eps = 1e-6

    # ----------------- Main Metrics -----------------
    dice = 2 * TP / (2 * TP + FP + FN + eps)
    iou = TP / (TP + FP + FN + eps)
    prec = TP / (TP + FP + eps)
    rec = TP / (TP + FN + eps)
    acc = (TP + TN) / (TP + FP + FN + TN + eps)
    spec = TN / (TN + FP + eps)

    # ----------------- Prevalence -----------------
    total_pixels = TP + TN + FP + FN + eps
    prevalence = (TP + FN) / total_pixels

    # ----------------- HD95 -----------------
    final_hd = (
        np.percentile(hd_list, 95)
        if calc_hd and len(hd_list) > 0
        else 0.0
    )

    # ----------------- AUC, AP, Recall@95% Specificity -----------------
    all_probs = (
        np.concatenate(probs_list)
        if len(probs_list)
        else np.array([], dtype=np.float32)
    )
    all_gts = (
        np.concatenate(gts_list)
        if len(gts_list)
        else np.array([], dtype=np.float32)
    )

    if len(all_probs) > 2_000_000:
        idx = np.random.choice(len(all_probs), 2_000_000, replace=False)
        all_probs, all_gts = all_probs[idx], all_gts[idx]

    try:
        # ROC-AUC is undefined when the evaluation data contain one class.
        auc = roc_auc_score(all_gts, all_probs)
        ap = average_precision_score(all_gts, all_probs)

        fpr, tpr, _ = roc_curve(all_gts, all_probs)
        valid_idx = np.where(fpr <= 0.05)[0]
        recall_at_high_spec = tpr[valid_idx[-1]] if len(valid_idx) > 0 else 0.0

    except ValueError:
        auc = 0.5
        ap = 0.0
        recall_at_high_spec = 0.0

    return {
        "loss": total_loss / max(n_batches, 1),

        # Main metrics
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(prec),
        "recall": float(rec),
        "accuracy": float(acc),
        "specificity": float(spec),

        # Discrimination metrics
        "auc": float(auc),
        "average_precision": float(ap),

        # Spatial metric
        "hd95": float(final_hd),

        # Prevalence and high-specificity operating point
        "prevalence": float(prevalence),
        "recall_at_95_spec": float(recall_at_high_spec),

        # Confusion matrix
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
    }


Cell 7 - Training

#SEEDS = 41

In [7]:

# Cell 7: Main Ablation Study (Dice@0.5 Checkpoint Selection)

import os
import gc
import time
import warnings
import pandas as pd
import torch
import numpy as np

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# Setup
# =====================================================================
os.makedirs("saved_models_newXAI", exist_ok=True)

EXPERIMENTS = [
    dict(name="2D_Baseline",  use_25d=False, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_Baseline", use_25d=True, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_DecAtt",   use_25d=True, enc_att=False, dec_att=True,  use_crop=False),
    dict(name="2.5D_EncAtt",   use_25d=True, enc_att=True,  dec_att=False, use_crop=False),
    dict(name="2.5D_FullAtt",  use_25d=True, enc_att=True,  dec_att=True,  use_crop=False),
]

# Only these two models will be evaluated on the Test set.
TEST_EXPERIMENTS = {
    "2D_Baseline",
    "2.5D_DecAtt"
}

train_logs = []
results_df = []
test_results_df = []

print(f"🚀 Starting Ablation Study with {len(EXPERIMENTS)} experiments...")

for i, cfg in enumerate(EXPERIMENTS):

    print(f"\n{'='*60}")
    print(f"🏆 Run {i+1}/{len(EXPERIMENTS)} : {cfg['name']}")
    print(f"{'='*60}")

    set_seed(SEED)

    current_train_ids = train_ids[:int(len(train_ids) * DATA_PERCENT)]
    current_val_ids = val_ids[:int(len(val_ids) * DATA_PERCENT)]
    current_test_ids = test_ids[:int(len(test_ids) * DATA_PERCENT)]

    dl_tr, dl_va, dl_te = make_loaders_ablation(
        current_train_ids,
        current_val_ids,
        current_test_ids,
        use_25d=cfg["use_25d"],
        use_crop=cfg["use_crop"]
    )

    in_channels = IN_CHANNELS if cfg["use_25d"] else 1

    model = SwinUNetAdaptive_DS(
        in_ch=in_channels,
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    ).to(device)

    loss_fn = ComboDiceTverskyLoss().to(device)

    # =================================================================
    # Optimizer with Differential Learning Rates
    # =================================================================
    base_lr = 3e-4
    encoder_params = []
    attention_params = []
    decoder_and_head_params = []
    scale_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # Separate scale parameters from other parameter groups.
        if "scale" in name:
            scale_params.append(param)
        elif any(key in name for key in [
            "enc_att", "dec_att", "local_attn", "global_attn", "gn"
        ]):
            attention_params.append(param)
        elif "encoder" in name:
            encoder_params.append(param)
        else:
            decoder_and_head_params.append(param)

    # Assign the original learning rates to each parameter group.
    param_groups = [
        {"params": encoder_params,          "lr": base_lr / 10},
        {"params": decoder_and_head_params, "lr": base_lr},
        {"params": attention_params,        "lr": base_lr * 2},
        {"params": scale_params,            "lr": 1e-3},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler()

    # =================================================================
    # Checkpointing Based on Validation Dice@0.5
    # =================================================================
    best_val_dice_05 = -1.0

    # Include the seed in the filename to avoid overwriting checkpoints
    # from different seed runs.
    best_path = f"saved_models_newXAI/best_model_{cfg['name']}_S{SEED}.pt"

    patience = 10
    patience_counter = 0
    start_time = time.time()

    for ep in range(1, EPOCHS + 1):

        tr_loss = train_one_epoch(
            model, dl_tr, optimizer, loss_fn, scaler, device
        )

        val_metrics_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=False, thr=0.5
        )

        val_loss = val_metrics_05["loss"]
        val_dice_05 = val_metrics_05["dice"]

        scheduler.step(val_dice_05)

        if val_dice_05 > best_val_dice_05:
            best_val_dice_05 = val_dice_05

            torch.save(
                {
                    "epoch": ep,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_dice": best_val_dice_05,
                    "config": cfg,
                },
                best_path
            )

            patience_counter = 0
        else:
            patience_counter += 1

        train_logs.append({
            "Experiment": cfg["name"],
            "Epoch": ep,
            "LR": optimizer.param_groups[1]["lr"],
            "Train_Loss": tr_loss,
            "Val_Loss": val_loss,
            "Val_Dice@0.5": val_dice_05,
            "Best_Dice@0.5": best_val_dice_05,
        })

        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            current_lr = optimizer.param_groups[1]["lr"]

            print(
                f"   Ep {ep:03d}/{EPOCHS} | LR: {current_lr:.2e} | "
                f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Val Dice@0.5: {val_dice_05:.4f} | "
                f"Best@0.5: {best_val_dice_05:.4f}"
            )

        if patience_counter >= patience:
            print(
                f"   🛑 Early stopping triggered! "
                f"No improvement for {patience} epochs."
            )
            break

    print(
        f"✅ Training {cfg['name']} finished in "
        f"{(time.time() - start_time) / 60:.1f} min."
    )

    # =================================================================
    # Load the Best Checkpoint Selected by Validation Dice@0.5
    # =================================================================
    checkpoint = torch.load(best_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict) and "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    print("⏳ Calculating final Train/Validation metrics (@0.5 only)...")

    with torch.no_grad():

        final_tr_05 = eval_full_split(
            model, dl_tr, loss_fn, device, calc_hd=False, thr=0.5
        )

        final_va_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=True, thr=0.5
        )

    print(
        f"   Final Dice Train/Val @0.5: "
        f"{final_tr_05.get('dice', 0):.4f} / "
        f"{final_va_05.get('dice', 0):.4f}"
    )

    # =================================================================
    # Save Training and Validation Results at Threshold 0.5
    # =================================================================
    for split_name, metrics_dict in [
        ("Train", final_tr_05),
        ("Val", final_va_05)
    ]:

        row = cfg.copy()
        row["seed"] = SEED
        row["split"] = split_name
        row["eval_setting"] = "Dice@0.5"
        row["thr_used"] = 0.5
        row.update(metrics_dict)

        if "loss" in row:
            del row["loss"]

        row["best_val_dice@0.5"] = best_val_dice_05

        results_df.append(row)

    # =================================================================
    # Test Evaluation: Only Baseline and DecAtt
    # =================================================================
    # The checkpoint was selected using Validation Dice@0.5.
    # Test data are not used for checkpoint or model selection.
    # =================================================================
    if cfg["name"] in TEST_EXPERIMENTS:

        print(f"\n🧪 Evaluating {cfg['name']} on the Test set...")

        with torch.no_grad():

            final_te_05 = eval_full_split(
                model, dl_te, loss_fn, device, calc_hd=True, thr=0.5
            )

        print(
            f"   Test Dice@0.5: "
            f"{final_te_05.get('dice', 0):.4f}"
        )

        test_row = cfg.copy()
        test_row["seed"] = SEED
        test_row["split"] = "Test"
        test_row["eval_setting"] = "Dice@0.5"
        test_row["thr_used"] = 0.5
        test_row.update(final_te_05)

        if "loss" in test_row:
            del test_row["loss"]

        test_row["best_val_dice@0.5"] = best_val_dice_05

        test_results_df.append(test_row)

    # =================================================================
    # Release resources before the next experiment
    # =================================================================
    del model, optimizer, scheduler, scaler, loss_fn
    del dl_tr, dl_va, dl_te

    torch.cuda.empty_cache()
    gc.collect()


# =====================================================================
# Compile and Save Main Train/Validation Results
# =====================================================================
print("\n" + "="*60)
print("🎯 FINAL TRAINING AND VALIDATION RESULTS:")
print("="*60)

df = pd.DataFrame(results_df)

# Exclude these metrics from the displayed and saved tables.
# Their calculations remain unchanged.
excluded_metrics = [
    "hd95",
    "recall_at_95_spec",
    "prevalence"
]

df = df.drop(columns=excluded_metrics, errors="ignore")

cols = list(df.columns)

preferred_order = [
    "name", "use_25d", "enc_att", "dec_att", "use_crop", "seed", "split",
    "eval_setting", "thr_used",

    # Main metrics
    "dice", "iou", "precision", "recall", "accuracy", "specificity",

    # Discrimination metrics
    "auc", "average_precision",

    # Confusion matrix
    "TP", "TN", "FP", "FN",

    # Training bookkeeping
    "best_val_dice@0.5"
]

final_cols = [c for c in preferred_order if c in cols]
final_cols += [c for c in cols if c not in preferred_order]

df = df[final_cols]

csv_path = (
    f"Final_Ablation_Results_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
)

df.to_csv(csv_path, index=False)

display(df)

print(f"💾 Train/Validation results saved: {csv_path}")


# =====================================================================
# Compile and Save Separate Test Results
# =====================================================================
print("\n" + "="*60)
print("🧪 FINAL TEST RESULTS (BASELINE AND DECATT):")
print("="*60)

if test_results_df:

    test_df = pd.DataFrame(test_results_df)

    # Exclude the same three metrics from the Test table.
    test_df = test_df.drop(
        columns=excluded_metrics,
        errors="ignore"
    )

    test_preferred_order = [
        "name", "use_25d", "enc_att", "dec_att", "use_crop",
        "seed", "split", "eval_setting", "thr_used",

        # Main metrics
        "dice", "iou", "precision", "recall", "accuracy", "specificity",

        # Discrimination metrics
        "auc", "average_precision",

        # Confusion matrix
        "TP", "TN", "FP", "FN",

        # Training bookkeeping
        "best_val_dice@0.5"
    ]

    test_cols = list(test_df.columns)

    final_test_cols = [
        c for c in test_preferred_order if c in test_cols
    ]

    final_test_cols += [
        c for c in test_cols if c not in test_preferred_order
    ]

    test_df = test_df[final_test_cols]

    test_csv_path = (
        f"Final_Test_Results_Baseline_DecAtt_"
        f"patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
    )

    test_df.to_csv(test_csv_path, index=False)

    display(test_df)

    print(f"💾 Test results saved: {test_csv_path}")

else:
    print("No Test results were generated.")


# =====================================================================
# Save Training Logs
# =====================================================================
if train_logs:

    train_log_df = pd.DataFrame(train_logs)

    log_path = (
        f"Training_Logs_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.xlsx"
    )

    train_log_df.to_excel(log_path, index=False)

    print(f"🧾 Training logs saved: {log_path}")


🚀 Starting Ablation Study with 5 experiments...

🏆 Run 1/5 : 2D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Train Loss: 0.2794 | Val Loss: 0.2198 | Val Dice@0.5: 0.6387 | Best@0.5: 0.6387
   Ep 005/100 | LR: 3.00e-04 | Train Loss: 0.1178 | Val Loss: 0.2234 | Val Dice@0.5: 0.6524 | Best@0.5: 0.6526
   Ep 010/100 | LR: 3.00e-04 | Train Loss: 0.0931 | Val Loss: 0.2162 | Val Dice@0.5: 0.6536 | Best@0.5: 0.6822
   Ep 015/100 | LR: 3.00e-04 | Train Loss: 0.0811 | Val Loss: 0.2136 | Val Dice@0.5: 0.6577 | Best@0.5: 0.6888
   Ep 020/100 | LR: 1.50e-04 | Train Loss: 0.0709 | Val Loss: 0.2294 | Val Dice@0.5: 0.6430 | Best@0.5: 0.6888
   🛑 Early stopping triggered! No improvement for 10 epochs.
✅ Training 2D_Baseline finished in 62.6 min.
⏳ Calculating final Train/Validation metrics (@0.5 only)...
   Final Dice Train/Val @0.5: 0.8106 / 0.6888

🧪 Evaluating 2D_Baseline on the Test set...
   Test Dice@0.5: 0.6867

🏆 Run 2/5 : 2.5D_Ba

,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,41,Train,Dice@0.5,0.5,0.810631,...,0.869477,0.999476,0.999644,0.995232,0.802434,1131122,1005956401,358677,169800,0.688755
1,2D_Baseline,False,False,False,False,41,Val,Dice@0.5,0.5,0.688755,...,0.630102,0.999189,0.999716,0.970047,0.674913,191219,212832759,60568,112254,0.688755
2,2.5D_Baseline,True,False,False,False,41,Train,Dice@0.5,0.5,0.846832,...,0.908290,0.999579,0.999696,0.995609,0.853777,1171718,1006020421,305552,118309,0.713691
3,2.5D_Baseline,True,False,False,False,41,Val,Dice@0.5,0.5,0.713691,...,0.637974,0.999271,0.999786,0.943434,0.692281,193608,212847854,45473,109865,0.713691
4,2.5D_DecAtt,True,False,True,False,41,Train,Dice@0.5,0.5,0.840780,...,0.901424,0.999555,0.999683,0.992827,0.853912,1182728,1005985323,318611,129338,0.765067
5,2.5D_DecAtt,True,False,True,False,41,Val,Dice@0.5,0.5,0.765067,...,0.733416,0.999359,0.999738,0.935334,0.723567,222572,212837535,55792,80901,0.765067
6,2.5D_EncAtt,True,True,False,False,41,Train,Dice@0.5,0.5,0.837762,...,0.902966,0.999551,0.999675,0.996971,0.858056,1168093,1005995490,326892,125525,0.729426
7,2.5D_EncAtt,True,True,False,False,41,Val,Dice@0.5,0.5,0.729426,...,0.684700,0.999277,0.999725,0.956441,0.710346,207788,212834858,58469,95685,0.729426
8,2.5D_FullAtt,True,True,True,False,41,Train,Dice@0.5,0.5,0.861041,...,0.913432,0.999612,0.999726,0.989757,0.862524,1210824,1006014358,276066,114752,0.776347
9,2.5D_FullAtt,True,True,True,False,41,Val,Dice@0.5,0.5,0.776347,...,0.733205,0.999399,0.999778,0.932028,0.748552,222508,212846090,47237,80965,0.776347


💾 Train/Validation results saved: Final_Ablation_Results_patch320_PNG_S41.csv

🧪 FINAL TEST RESULTS (BASELINE AND DECATT):


,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,41,Test,Dice@0.5,0.5,0.686719,...,0.616977,0.999290,0.999773,0.971199,0.679866,160670,206131135,46850,99745,0.688755
1,2.5D_DecAtt,True,False,True,False,41,Test,Dice@0.5,0.5,0.721517,...,0.680456,0.999337,0.999740,0.932898,0.687262,177201,206124411,53574,83214,0.765067


💾 Test results saved: Final_Test_Results_Baseline_DecAtt_patch320_PNG_S41.csv
🧾 Training logs saved: Training_Logs_patch320_PNG_S41.xlsx


#SEEDS = 42

In [7]:

# Cell 7: Main Ablation Study (Dice@0.5 Checkpoint Selection)

import os
import gc
import time
import warnings
import pandas as pd
import torch
import numpy as np

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# Setup
# =====================================================================
os.makedirs("saved_models_newXAI", exist_ok=True)

EXPERIMENTS = [
    dict(name="2D_Baseline",  use_25d=False, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_Baseline", use_25d=True, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_DecAtt",   use_25d=True, enc_att=False, dec_att=True,  use_crop=False),
    dict(name="2.5D_EncAtt",   use_25d=True, enc_att=True,  dec_att=False, use_crop=False),
    dict(name="2.5D_FullAtt",  use_25d=True, enc_att=True,  dec_att=True,  use_crop=False),
]

# Only these two models will be evaluated on the Test set.
TEST_EXPERIMENTS = {
    "2D_Baseline",
    "2.5D_DecAtt"
}

train_logs = []
results_df = []
test_results_df = []

print(f"🚀 Starting Ablation Study with {len(EXPERIMENTS)} experiments...")

for i, cfg in enumerate(EXPERIMENTS):

    print(f"\n{'='*60}")
    print(f"🏆 Run {i+1}/{len(EXPERIMENTS)} : {cfg['name']}")
    print(f"{'='*60}")

    set_seed(SEED)

    current_train_ids = train_ids[:int(len(train_ids) * DATA_PERCENT)]
    current_val_ids = val_ids[:int(len(val_ids) * DATA_PERCENT)]
    current_test_ids = test_ids[:int(len(test_ids) * DATA_PERCENT)]

    dl_tr, dl_va, dl_te = make_loaders_ablation(
        current_train_ids,
        current_val_ids,
        current_test_ids,
        use_25d=cfg["use_25d"],
        use_crop=cfg["use_crop"]
    )

    in_channels = IN_CHANNELS if cfg["use_25d"] else 1

    model = SwinUNetAdaptive_DS(
        in_ch=in_channels,
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    ).to(device)

    loss_fn = ComboDiceTverskyLoss().to(device)

    # =================================================================
    # Optimizer with Differential Learning Rates
    # =================================================================
    base_lr = 3e-4
    encoder_params = []
    attention_params = []
    decoder_and_head_params = []
    scale_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # Separate scale parameters from other parameter groups.
        if "scale" in name:
            scale_params.append(param)
        elif any(key in name for key in [
            "enc_att", "dec_att", "local_attn", "global_attn", "gn"
        ]):
            attention_params.append(param)
        elif "encoder" in name:
            encoder_params.append(param)
        else:
            decoder_and_head_params.append(param)

    # Assign the original learning rates to each parameter group.
    param_groups = [
        {"params": encoder_params,          "lr": base_lr / 10},
        {"params": decoder_and_head_params, "lr": base_lr},
        {"params": attention_params,        "lr": base_lr * 2},
        {"params": scale_params,            "lr": 1e-3},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler()

    # =================================================================
    # Checkpointing Based on Validation Dice@0.5
    # =================================================================
    best_val_dice_05 = -1.0

    # Include the seed in the filename to avoid overwriting checkpoints
    # from different seed runs.
    best_path = f"saved_models_newXAI/best_model_{cfg['name']}_S{SEED}.pt"

    patience = 10
    patience_counter = 0
    start_time = time.time()

    for ep in range(1, EPOCHS + 1):

        tr_loss = train_one_epoch(
            model, dl_tr, optimizer, loss_fn, scaler, device
        )

        val_metrics_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=False, thr=0.5
        )

        val_loss = val_metrics_05["loss"]
        val_dice_05 = val_metrics_05["dice"]

        scheduler.step(val_dice_05)

        if val_dice_05 > best_val_dice_05:
            best_val_dice_05 = val_dice_05

            torch.save(
                {
                    "epoch": ep,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_dice": best_val_dice_05,
                    "config": cfg,
                },
                best_path
            )

            patience_counter = 0
        else:
            patience_counter += 1

        train_logs.append({
            "Experiment": cfg["name"],
            "Epoch": ep,
            "LR": optimizer.param_groups[1]["lr"],
            "Train_Loss": tr_loss,
            "Val_Loss": val_loss,
            "Val_Dice@0.5": val_dice_05,
            "Best_Dice@0.5": best_val_dice_05,
        })

        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            current_lr = optimizer.param_groups[1]["lr"]

            print(
                f"   Ep {ep:03d}/{EPOCHS} | LR: {current_lr:.2e} | "
                f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Val Dice@0.5: {val_dice_05:.4f} | "
                f"Best@0.5: {best_val_dice_05:.4f}"
            )

        if patience_counter >= patience:
            print(
                f"   🛑 Early stopping triggered! "
                f"No improvement for {patience} epochs."
            )
            break

    print(
        f"✅ Training {cfg['name']} finished in "
        f"{(time.time() - start_time) / 60:.1f} min."
    )

    # =================================================================
    # Load the Best Checkpoint Selected by Validation Dice@0.5
    # =================================================================
    checkpoint = torch.load(best_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict) and "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    print("⏳ Calculating final Train/Validation metrics (@0.5 only)...")

    with torch.no_grad():

        final_tr_05 = eval_full_split(
            model, dl_tr, loss_fn, device, calc_hd=False, thr=0.5
        )

        final_va_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=True, thr=0.5
        )

    print(
        f"   Final Dice Train/Val @0.5: "
        f"{final_tr_05.get('dice', 0):.4f} / "
        f"{final_va_05.get('dice', 0):.4f}"
    )

    # =================================================================
    # Save Training and Validation Results at Threshold 0.5
    # =================================================================
    for split_name, metrics_dict in [
        ("Train", final_tr_05),
        ("Val", final_va_05)
    ]:

        row = cfg.copy()
        row["seed"] = SEED
        row["split"] = split_name
        row["eval_setting"] = "Dice@0.5"
        row["thr_used"] = 0.5
        row.update(metrics_dict)

        if "loss" in row:
            del row["loss"]

        row["best_val_dice@0.5"] = best_val_dice_05

        results_df.append(row)

    # =================================================================
    # Test Evaluation: Only Baseline and DecAtt
    # =================================================================
    # The checkpoint was selected using Validation Dice@0.5.
    # Test data are not used for checkpoint or model selection.
    # =================================================================
    if cfg["name"] in TEST_EXPERIMENTS:

        print(f"\n🧪 Evaluating {cfg['name']} on the Test set...")

        with torch.no_grad():

            final_te_05 = eval_full_split(
                model, dl_te, loss_fn, device, calc_hd=True, thr=0.5
            )

        print(
            f"   Test Dice@0.5: "
            f"{final_te_05.get('dice', 0):.4f}"
        )

        test_row = cfg.copy()
        test_row["seed"] = SEED
        test_row["split"] = "Test"
        test_row["eval_setting"] = "Dice@0.5"
        test_row["thr_used"] = 0.5
        test_row.update(final_te_05)

        if "loss" in test_row:
            del test_row["loss"]

        test_row["best_val_dice@0.5"] = best_val_dice_05

        test_results_df.append(test_row)

    # =================================================================
    # Release resources before the next experiment
    # =================================================================
    del model, optimizer, scheduler, scaler, loss_fn
    del dl_tr, dl_va, dl_te

    torch.cuda.empty_cache()
    gc.collect()


# =====================================================================
# Compile and Save Main Train/Validation Results
# =====================================================================
print("\n" + "="*60)
print("🎯 FINAL TRAINING AND VALIDATION RESULTS:")
print("="*60)

df = pd.DataFrame(results_df)

# Exclude these metrics from the displayed and saved tables.
# Their calculations remain unchanged.
excluded_metrics = [
    "hd95",
    "recall_at_95_spec",
    "prevalence"
]

df = df.drop(columns=excluded_metrics, errors="ignore")

cols = list(df.columns)

preferred_order = [
    "name", "use_25d", "enc_att", "dec_att", "use_crop", "seed", "split",
    "eval_setting", "thr_used",

    # Main metrics
    "dice", "iou", "precision", "recall", "accuracy", "specificity",

    # Discrimination metrics
    "auc", "average_precision",

    # Confusion matrix
    "TP", "TN", "FP", "FN",

    # Training bookkeeping
    "best_val_dice@0.5"
]

final_cols = [c for c in preferred_order if c in cols]
final_cols += [c for c in cols if c not in preferred_order]

df = df[final_cols]

csv_path = (
    f"Final_Ablation_Results_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
)

df.to_csv(csv_path, index=False)

display(df)

print(f"💾 Train/Validation results saved: {csv_path}")


# =====================================================================
# Compile and Save Separate Test Results
# =====================================================================
print("\n" + "="*60)
print("🧪 FINAL TEST RESULTS (BASELINE AND DECATT):")
print("="*60)

if test_results_df:

    test_df = pd.DataFrame(test_results_df)

    # Exclude the same three metrics from the Test table.
    test_df = test_df.drop(
        columns=excluded_metrics,
        errors="ignore"
    )

    test_preferred_order = [
        "name", "use_25d", "enc_att", "dec_att", "use_crop",
        "seed", "split", "eval_setting", "thr_used",

        # Main metrics
        "dice", "iou", "precision", "recall", "accuracy", "specificity",

        # Discrimination metrics
        "auc", "average_precision",

        # Confusion matrix
        "TP", "TN", "FP", "FN",

        # Training bookkeeping
        "best_val_dice@0.5"
    ]

    test_cols = list(test_df.columns)

    final_test_cols = [
        c for c in test_preferred_order if c in test_cols
    ]

    final_test_cols += [
        c for c in test_cols if c not in test_preferred_order
    ]

    test_df = test_df[final_test_cols]

    test_csv_path = (
        f"Final_Test_Results_Baseline_DecAtt_"
        f"patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
    )

    test_df.to_csv(test_csv_path, index=False)

    display(test_df)

    print(f"💾 Test results saved: {test_csv_path}")

else:
    print("No Test results were generated.")


# =====================================================================
# Save Training Logs
# =====================================================================
if train_logs:

    train_log_df = pd.DataFrame(train_logs)

    log_path = (
        f"Training_Logs_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.xlsx"
    )

    train_log_df.to_excel(log_path, index=False)

    print(f"🧾 Training logs saved: {log_path}")


🚀 Starting Ablation Study with 5 experiments...

🏆 Run 1/5 : 2D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Train Loss: 0.2810 | Val Loss: 0.2382 | Val Dice@0.5: 0.6158 | Best@0.5: 0.6158
   Ep 005/100 | LR: 3.00e-04 | Train Loss: 0.1161 | Val Loss: 0.2327 | Val Dice@0.5: 0.6256 | Best@0.5: 0.6595
   Ep 010/100 | LR: 3.00e-04 | Train Loss: 0.0904 | Val Loss: 0.2123 | Val Dice@0.5: 0.6708 | Best@0.5: 0.6939
   Ep 015/100 | LR: 1.50e-04 | Train Loss: 0.0780 | Val Loss: 0.2047 | Val Dice@0.5: 0.6782 | Best@0.5: 0.6939
   🛑 Early stopping triggered! No improvement for 10 epochs.
✅ Training 2D_Baseline finished in 52.6 min.
⏳ Calculating final Train/Validation metrics (@0.5 only)...
   Final Dice Train/Val @0.5: 0.8093 / 0.6939

🧪 Evaluating 2D_Baseline on the Test set...
   Test Dice@0.5: 0.7030

🏆 Run 2/5 : 2.5D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Tra

,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,42,Train,Dice@0.5,0.5,0.809271,...,0.876150,0.999466,0.999625,0.990054,0.786148,1142215,1005935392,376933,161460,0.693908
1,2D_Baseline,False,False,False,False,42,Val,Dice@0.5,0.5,0.693908,...,0.659653,0.999172,0.999656,0.940343,0.657490,200187,212820003,73324,103286,0.693908
2,2.5D_Baseline,True,False,False,False,42,Train,Dice@0.5,0.5,0.837075,...,0.900131,0.999551,0.999679,0.996224,0.844986,1161391,1006002511,323243,128855,0.734842
3,2.5D_Baseline,True,False,False,False,42,Val,Dice@0.5,0.5,0.734842,...,0.679540,0.999302,0.999758,0.930942,0.701874,206222,212841753,51574,97251,0.734842
4,2.5D_DecAtt,True,False,True,False,42,Train,Dice@0.5,0.5,0.830962,...,0.871835,0.999535,0.999703,0.989752,0.846858,1150898,1005996862,299051,169189,0.742879
5,2.5D_DecAtt,True,False,True,False,42,Val,Dice@0.5,0.5,0.742879,...,0.695492,0.999315,0.999748,0.924915,0.709198,211063,212839633,53694,92410,0.742879
6,2.5D_EncAtt,True,True,False,False,42,Train,Dice@0.5,0.5,0.834113,...,0.892632,0.999541,0.999679,0.995054,0.839625,1162801,1005990687,322647,139865,0.730293
7,2.5D_EncAtt,True,True,False,False,42,Val,Dice@0.5,0.5,0.730293,...,0.677200,0.999288,0.999747,0.940904,0.721837,205512,212839491,53836,97961,0.730293
8,2.5D_FullAtt,True,True,True,False,42,Train,Dice@0.5,0.5,0.857335,...,0.909411,0.999609,0.999726,0.994913,0.856815,1183438,1006038701,275976,117885,0.766378
9,2.5D_FullAtt,True,True,True,False,42,Val,Dice@0.5,0.5,0.766378,...,0.734177,0.999363,0.999741,0.953469,0.725332,222803,212838159,55168,80670,0.766378


💾 Train/Validation results saved: Final_Ablation_Results_patch320_PNG_S42.csv

🧪 FINAL TEST RESULTS (BASELINE AND DECATT):


,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,42,Test,Dice@0.5,0.5,0.702981,...,0.672868,0.999283,0.999695,0.957106,0.686943,175225,206115105,62880,85190,0.693908
1,2.5D_DecAtt,True,False,True,False,42,Test,Dice@0.5,0.5,0.690704,...,0.629649,0.999289,0.999756,0.922027,0.654763,163970,206127579,50406,96445,0.742879


💾 Test results saved: Final_Test_Results_Baseline_DecAtt_patch320_PNG_S42.csv
🧾 Training logs saved: Training_Logs_patch320_PNG_S42.xlsx


#SEEDS = 44

In [7]:

# Cell 7: Main Ablation Study (Dice@0.5 Checkpoint Selection)

import os
import gc
import time
import warnings
import pandas as pd
import torch
import numpy as np

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# Setup
# =====================================================================
os.makedirs("saved_models_newXAI", exist_ok=True)

EXPERIMENTS = [
    dict(name="2D_Baseline",  use_25d=False, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_Baseline", use_25d=True, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_DecAtt",   use_25d=True, enc_att=False, dec_att=True,  use_crop=False),
    dict(name="2.5D_EncAtt",   use_25d=True, enc_att=True,  dec_att=False, use_crop=False),
    dict(name="2.5D_FullAtt",  use_25d=True, enc_att=True,  dec_att=True,  use_crop=False),
]

# Only these two models will be evaluated on the Test set.
TEST_EXPERIMENTS = {
    "2D_Baseline",
    "2.5D_DecAtt"
}

train_logs = []
results_df = []
test_results_df = []

print(f"🚀 Starting Ablation Study with {len(EXPERIMENTS)} experiments...")

for i, cfg in enumerate(EXPERIMENTS):

    print(f"\n{'='*60}")
    print(f"🏆 Run {i+1}/{len(EXPERIMENTS)} : {cfg['name']}")
    print(f"{'='*60}")

    set_seed(SEED)

    current_train_ids = train_ids[:int(len(train_ids) * DATA_PERCENT)]
    current_val_ids = val_ids[:int(len(val_ids) * DATA_PERCENT)]
    current_test_ids = test_ids[:int(len(test_ids) * DATA_PERCENT)]

    dl_tr, dl_va, dl_te = make_loaders_ablation(
        current_train_ids,
        current_val_ids,
        current_test_ids,
        use_25d=cfg["use_25d"],
        use_crop=cfg["use_crop"]
    )

    in_channels = IN_CHANNELS if cfg["use_25d"] else 1

    model = SwinUNetAdaptive_DS(
        in_ch=in_channels,
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    ).to(device)

    loss_fn = ComboDiceTverskyLoss().to(device)

    # =================================================================
    # Optimizer with Differential Learning Rates
    # =================================================================
    base_lr = 3e-4
    encoder_params = []
    attention_params = []
    decoder_and_head_params = []
    scale_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # Separate scale parameters from other parameter groups.
        if "scale" in name:
            scale_params.append(param)
        elif any(key in name for key in [
            "enc_att", "dec_att", "local_attn", "global_attn", "gn"
        ]):
            attention_params.append(param)
        elif "encoder" in name:
            encoder_params.append(param)
        else:
            decoder_and_head_params.append(param)

    # Assign the original learning rates to each parameter group.
    param_groups = [
        {"params": encoder_params,          "lr": base_lr / 10},
        {"params": decoder_and_head_params, "lr": base_lr},
        {"params": attention_params,        "lr": base_lr * 2},
        {"params": scale_params,            "lr": 1e-3},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler()

    # =================================================================
    # Checkpointing Based on Validation Dice@0.5
    # =================================================================
    best_val_dice_05 = -1.0

    # Include the seed in the filename to avoid overwriting checkpoints
    # from different seed runs.
    best_path = f"saved_models_newXAI/best_model_{cfg['name']}_S{SEED}.pt"

    patience = 10
    patience_counter = 0
    start_time = time.time()

    for ep in range(1, EPOCHS + 1):

        tr_loss = train_one_epoch(
            model, dl_tr, optimizer, loss_fn, scaler, device
        )

        val_metrics_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=False, thr=0.5
        )

        val_loss = val_metrics_05["loss"]
        val_dice_05 = val_metrics_05["dice"]

        scheduler.step(val_dice_05)

        if val_dice_05 > best_val_dice_05:
            best_val_dice_05 = val_dice_05

            torch.save(
                {
                    "epoch": ep,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_dice": best_val_dice_05,
                    "config": cfg,
                },
                best_path
            )

            patience_counter = 0
        else:
            patience_counter += 1

        train_logs.append({
            "Experiment": cfg["name"],
            "Epoch": ep,
            "LR": optimizer.param_groups[1]["lr"],
            "Train_Loss": tr_loss,
            "Val_Loss": val_loss,
            "Val_Dice@0.5": val_dice_05,
            "Best_Dice@0.5": best_val_dice_05,
        })

        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            current_lr = optimizer.param_groups[1]["lr"]

            print(
                f"   Ep {ep:03d}/{EPOCHS} | LR: {current_lr:.2e} | "
                f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Val Dice@0.5: {val_dice_05:.4f} | "
                f"Best@0.5: {best_val_dice_05:.4f}"
            )

        if patience_counter >= patience:
            print(
                f"   🛑 Early stopping triggered! "
                f"No improvement for {patience} epochs."
            )
            break

    print(
        f"✅ Training {cfg['name']} finished in "
        f"{(time.time() - start_time) / 60:.1f} min."
    )

    # =================================================================
    # Load the Best Checkpoint Selected by Validation Dice@0.5
    # =================================================================
    checkpoint = torch.load(best_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict) and "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    print("⏳ Calculating final Train/Validation metrics (@0.5 only)...")

    with torch.no_grad():

        final_tr_05 = eval_full_split(
            model, dl_tr, loss_fn, device, calc_hd=False, thr=0.5
        )

        final_va_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=True, thr=0.5
        )

    print(
        f"   Final Dice Train/Val @0.5: "
        f"{final_tr_05.get('dice', 0):.4f} / "
        f"{final_va_05.get('dice', 0):.4f}"
    )

    # =================================================================
    # Save Training and Validation Results at Threshold 0.5
    # =================================================================
    for split_name, metrics_dict in [
        ("Train", final_tr_05),
        ("Val", final_va_05)
    ]:

        row = cfg.copy()
        row["seed"] = SEED
        row["split"] = split_name
        row["eval_setting"] = "Dice@0.5"
        row["thr_used"] = 0.5
        row.update(metrics_dict)

        if "loss" in row:
            del row["loss"]

        row["best_val_dice@0.5"] = best_val_dice_05

        results_df.append(row)

    # =================================================================
    # Test Evaluation: Only Baseline and DecAtt
    # =================================================================
    # The checkpoint was selected using Validation Dice@0.5.
    # Test data are not used for checkpoint or model selection.
    # =================================================================
    if cfg["name"] in TEST_EXPERIMENTS:

        print(f"\n🧪 Evaluating {cfg['name']} on the Test set...")

        with torch.no_grad():

            final_te_05 = eval_full_split(
                model, dl_te, loss_fn, device, calc_hd=True, thr=0.5
            )

        print(
            f"   Test Dice@0.5: "
            f"{final_te_05.get('dice', 0):.4f}"
        )

        test_row = cfg.copy()
        test_row["seed"] = SEED
        test_row["split"] = "Test"
        test_row["eval_setting"] = "Dice@0.5"
        test_row["thr_used"] = 0.5
        test_row.update(final_te_05)

        if "loss" in test_row:
            del test_row["loss"]

        test_row["best_val_dice@0.5"] = best_val_dice_05

        test_results_df.append(test_row)

    # =================================================================
    # Release resources before the next experiment
    # =================================================================
    del model, optimizer, scheduler, scaler, loss_fn
    del dl_tr, dl_va, dl_te

    torch.cuda.empty_cache()
    gc.collect()


# =====================================================================
# Compile and Save Main Train/Validation Results
# =====================================================================
print("\n" + "="*60)
print("🎯 FINAL TRAINING AND VALIDATION RESULTS:")
print("="*60)

df = pd.DataFrame(results_df)

# Exclude these metrics from the displayed and saved tables.
# Their calculations remain unchanged.
excluded_metrics = [
    "hd95",
    "recall_at_95_spec",
    "prevalence"
]

df = df.drop(columns=excluded_metrics, errors="ignore")

cols = list(df.columns)

preferred_order = [
    "name", "use_25d", "enc_att", "dec_att", "use_crop", "seed", "split",
    "eval_setting", "thr_used",

    # Main metrics
    "dice", "iou", "precision", "recall", "accuracy", "specificity",

    # Discrimination metrics
    "auc", "average_precision",

    # Confusion matrix
    "TP", "TN", "FP", "FN",

    # Training bookkeeping
    "best_val_dice@0.5"
]

final_cols = [c for c in preferred_order if c in cols]
final_cols += [c for c in cols if c not in preferred_order]

df = df[final_cols]

csv_path = (
    f"Final_Ablation_Results_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
)

df.to_csv(csv_path, index=False)

display(df)

print(f"💾 Train/Validation results saved: {csv_path}")


# =====================================================================
# Compile and Save Separate Test Results
# =====================================================================
print("\n" + "="*60)
print("🧪 FINAL TEST RESULTS (BASELINE AND DECATT):")
print("="*60)

if test_results_df:

    test_df = pd.DataFrame(test_results_df)

    # Exclude the same three metrics from the Test table.
    test_df = test_df.drop(
        columns=excluded_metrics,
        errors="ignore"
    )

    test_preferred_order = [
        "name", "use_25d", "enc_att", "dec_att", "use_crop",
        "seed", "split", "eval_setting", "thr_used",

        # Main metrics
        "dice", "iou", "precision", "recall", "accuracy", "specificity",

        # Discrimination metrics
        "auc", "average_precision",

        # Confusion matrix
        "TP", "TN", "FP", "FN",

        # Training bookkeeping
        "best_val_dice@0.5"
    ]

    test_cols = list(test_df.columns)

    final_test_cols = [
        c for c in test_preferred_order if c in test_cols
    ]

    final_test_cols += [
        c for c in test_cols if c not in test_preferred_order
    ]

    test_df = test_df[final_test_cols]

    test_csv_path = (
        f"Final_Test_Results_Baseline_DecAtt_"
        f"patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
    )

    test_df.to_csv(test_csv_path, index=False)

    display(test_df)

    print(f"💾 Test results saved: {test_csv_path}")

else:
    print("No Test results were generated.")


# =====================================================================
# Save Training Logs
# =====================================================================
if train_logs:

    train_log_df = pd.DataFrame(train_logs)

    log_path = (
        f"Training_Logs_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.xlsx"
    )

    train_log_df.to_excel(log_path, index=False)

    print(f"🧾 Training logs saved: {log_path}")


🚀 Starting Ablation Study with 5 experiments...

🏆 Run 1/5 : 2D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Train Loss: 0.3703 | Val Loss: 0.2357 | Val Dice@0.5: 0.6133 | Best@0.5: 0.6133
   Ep 005/100 | LR: 3.00e-04 | Train Loss: 0.1182 | Val Loss: 0.2134 | Val Dice@0.5: 0.6628 | Best@0.5: 0.6875
   Ep 010/100 | LR: 3.00e-04 | Train Loss: 0.0940 | Val Loss: 0.1996 | Val Dice@0.5: 0.7080 | Best@0.5: 0.7080
   Ep 015/100 | LR: 3.00e-04 | Train Loss: 0.0810 | Val Loss: 0.2058 | Val Dice@0.5: 0.6808 | Best@0.5: 0.7080
   Ep 020/100 | LR: 1.50e-04 | Train Loss: 0.0711 | Val Loss: 0.2065 | Val Dice@0.5: 0.6836 | Best@0.5: 0.7080
   🛑 Early stopping triggered! No improvement for 10 epochs.
✅ Training 2D_Baseline finished in 103.8 min.
⏳ Calculating final Train/Validation metrics (@0.5 only)...
   Final Dice Train/Val @0.5: 0.8121 / 0.7080

🧪 Evaluating 2D_Baseline on the Test set...
   Test Dice@0.5: 0.6773

🏆 Run 2/5 : 2.5D_B

,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,44,Train,Dice@0.5,0.5,0.812103,...,0.869969,0.999482,0.999649,0.991848,0.801644,1126943,1005967572,353045,168440,0.708036
1,2D_Baseline,False,False,False,False,44,Val,Dice@0.5,0.5,0.708036,...,0.663950,0.999221,0.999698,0.942527,0.678058,201491,212829136,64191,101982,0.708036
2,2.5D_Baseline,True,False,False,False,44,Train,Dice@0.5,0.5,0.798564,...,0.873572,0.999425,0.999590,0.993316,0.808450,1147965,1005888890,413006,166139,0.710331
3,2.5D_Baseline,True,False,False,False,44,Val,Dice@0.5,0.5,0.710331,...,0.710063,0.999176,0.999588,0.951161,0.697678,215485,212805568,87759,87988,0.710331
4,2.5D_DecAtt,True,False,True,False,44,Train,Dice@0.5,0.5,0.845663,...,0.896932,0.999573,0.999707,0.993211,0.845908,1178953,1006006719,294853,135475,0.769652
5,2.5D_DecAtt,True,False,True,False,44,Val,Dice@0.5,0.5,0.769652,...,0.727534,0.999380,0.999768,0.940992,0.722746,220787,212843855,49472,82686,0.769652
6,2.5D_EncAtt,True,True,False,False,44,Train,Dice@0.5,0.5,0.820257,...,0.878260,0.999497,0.999656,0.991023,0.822021,1156898,1005952081,346658,160363,0.729116
7,2.5D_EncAtt,True,True,False,False,44,Val,Dice@0.5,0.5,0.729116,...,0.700833,0.999259,0.999684,0.961707,0.702659,212684,212826082,67245,90789,0.729116
8,2.5D_FullAtt,True,True,True,False,44,Train,Dice@0.5,0.5,0.854635,...,0.906272,0.999601,0.999722,0.994362,0.860145,1182471,1006031275,279961,122293,0.732579
9,2.5D_FullAtt,True,True,True,False,44,Val,Dice@0.5,0.5,0.732579,...,0.671440,0.999302,0.999770,0.927871,0.708002,203764,212844272,49055,99709,0.732579


💾 Train/Validation results saved: Final_Ablation_Results_patch320_PNG_S44.csv

🧪 FINAL TEST RESULTS (BASELINE AND DECATT):


,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,44,Test,Dice@0.5,0.5,0.677287,...,0.609051,0.999268,0.999761,0.944255,0.652182,158606,206128649,49336,101809,0.708036
1,2.5D_DecAtt,True,False,True,False,44,Test,Dice@0.5,0.5,0.711916,...,0.653684,0.999333,0.999769,0.932732,0.642684,170229,206130401,47584,90186,0.769652


💾 Test results saved: Final_Test_Results_Baseline_DecAtt_patch320_PNG_S44.csv
🧾 Training logs saved: Training_Logs_patch320_PNG_S44.xlsx


#SEED = 45

In [8]:

# Cell 7: Main Ablation Study (Dice@0.5 Checkpoint Selection)

import os
import gc
import time
import warnings
import pandas as pd
import torch
import numpy as np

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# Setup
# =====================================================================
os.makedirs("saved_models_newXAI", exist_ok=True)

EXPERIMENTS = [
    dict(name="2D_Baseline",  use_25d=False, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_Baseline", use_25d=True, enc_att=False, dec_att=False, use_crop=False),
    dict(name="2.5D_DecAtt",   use_25d=True, enc_att=False, dec_att=True,  use_crop=False),
    dict(name="2.5D_EncAtt",   use_25d=True, enc_att=True,  dec_att=False, use_crop=False),
    dict(name="2.5D_FullAtt",  use_25d=True, enc_att=True,  dec_att=True,  use_crop=False),
]

# Only these two models will be evaluated on the Test set.
TEST_EXPERIMENTS = {
    "2D_Baseline",
    "2.5D_DecAtt"
}

train_logs = []
results_df = []
test_results_df = []

print(f"🚀 Starting Ablation Study with {len(EXPERIMENTS)} experiments...")

for i, cfg in enumerate(EXPERIMENTS):

    print(f"\n{'='*60}")
    print(f"🏆 Run {i+1}/{len(EXPERIMENTS)} : {cfg['name']}")
    print(f"{'='*60}")

    set_seed(SEED)

    current_train_ids = train_ids[:int(len(train_ids) * DATA_PERCENT)]
    current_val_ids = val_ids[:int(len(val_ids) * DATA_PERCENT)]
    current_test_ids = test_ids[:int(len(test_ids) * DATA_PERCENT)]

    dl_tr, dl_va, dl_te = make_loaders_ablation(
        current_train_ids,
        current_val_ids,
        current_test_ids,
        use_25d=cfg["use_25d"],
        use_crop=cfg["use_crop"]
    )

    in_channels = IN_CHANNELS if cfg["use_25d"] else 1

    model = SwinUNetAdaptive_DS(
        in_ch=in_channels,
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    ).to(device)

    loss_fn = ComboDiceTverskyLoss().to(device)

    # =================================================================
    # Optimizer with Differential Learning Rates
    # =================================================================
    base_lr = 3e-4
    encoder_params = []
    attention_params = []
    decoder_and_head_params = []
    scale_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # Separate scale parameters from other parameter groups.
        if "scale" in name:
            scale_params.append(param)
        elif any(key in name for key in [
            "enc_att", "dec_att", "local_attn", "global_attn", "gn"
        ]):
            attention_params.append(param)
        elif "encoder" in name:
            encoder_params.append(param)
        else:
            decoder_and_head_params.append(param)

    # Assign the original learning rates to each parameter group.
    param_groups = [
        {"params": encoder_params,          "lr": base_lr / 10},
        {"params": decoder_and_head_params, "lr": base_lr},
        {"params": attention_params,        "lr": base_lr * 2},
        {"params": scale_params,            "lr": 1e-3},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler()

    # =================================================================
    # Checkpointing Based on Validation Dice@0.5
    # =================================================================
    best_val_dice_05 = -1.0

    # Include the seed in the filename to avoid overwriting checkpoints
    # from different seed runs.
    best_path = f"saved_models_newXAI/best_model_{cfg['name']}_S{SEED}.pt"

    patience = 10
    patience_counter = 0
    start_time = time.time()

    for ep in range(1, EPOCHS + 1):

        tr_loss = train_one_epoch(
            model, dl_tr, optimizer, loss_fn, scaler, device
        )

        val_metrics_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=False, thr=0.5
        )

        val_loss = val_metrics_05["loss"]
        val_dice_05 = val_metrics_05["dice"]

        scheduler.step(val_dice_05)

        if val_dice_05 > best_val_dice_05:
            best_val_dice_05 = val_dice_05

            torch.save(
                {
                    "epoch": ep,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_dice": best_val_dice_05,
                    "config": cfg,
                },
                best_path
            )

            patience_counter = 0
        else:
            patience_counter += 1

        train_logs.append({
            "Experiment": cfg["name"],
            "Epoch": ep,
            "LR": optimizer.param_groups[1]["lr"],
            "Train_Loss": tr_loss,
            "Val_Loss": val_loss,
            "Val_Dice@0.5": val_dice_05,
            "Best_Dice@0.5": best_val_dice_05,
        })

        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            current_lr = optimizer.param_groups[1]["lr"]

            print(
                f"   Ep {ep:03d}/{EPOCHS} | LR: {current_lr:.2e} | "
                f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Val Dice@0.5: {val_dice_05:.4f} | "
                f"Best@0.5: {best_val_dice_05:.4f}"
            )

        if patience_counter >= patience:
            print(
                f"   🛑 Early stopping triggered! "
                f"No improvement for {patience} epochs."
            )
            break

    print(
        f"✅ Training {cfg['name']} finished in "
        f"{(time.time() - start_time) / 60:.1f} min."
    )

    # =================================================================
    # Load the Best Checkpoint Selected by Validation Dice@0.5
    # =================================================================
    checkpoint = torch.load(best_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict) and "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    print("⏳ Calculating final Train/Validation metrics (@0.5 only)...")

    with torch.no_grad():

        final_tr_05 = eval_full_split(
            model, dl_tr, loss_fn, device, calc_hd=False, thr=0.5
        )

        final_va_05 = eval_full_split(
            model, dl_va, loss_fn, device, calc_hd=True, thr=0.5
        )

    print(
        f"   Final Dice Train/Val @0.5: "
        f"{final_tr_05.get('dice', 0):.4f} / "
        f"{final_va_05.get('dice', 0):.4f}"
    )

    # =================================================================
    # Save Training and Validation Results at Threshold 0.5
    # =================================================================
    for split_name, metrics_dict in [
        ("Train", final_tr_05),
        ("Val", final_va_05)
    ]:

        row = cfg.copy()
        row["seed"] = SEED
        row["split"] = split_name
        row["eval_setting"] = "Dice@0.5"
        row["thr_used"] = 0.5
        row.update(metrics_dict)

        if "loss" in row:
            del row["loss"]

        row["best_val_dice@0.5"] = best_val_dice_05

        results_df.append(row)

    # =================================================================
    # Test Evaluation: Only Baseline and DecAtt
    # =================================================================
    # The checkpoint was selected using Validation Dice@0.5.
    # Test data are not used for checkpoint or model selection.
    # =================================================================
    if cfg["name"] in TEST_EXPERIMENTS:

        print(f"\n🧪 Evaluating {cfg['name']} on the Test set...")

        with torch.no_grad():

            final_te_05 = eval_full_split(
                model, dl_te, loss_fn, device, calc_hd=True, thr=0.5
            )

        print(
            f"   Test Dice@0.5: "
            f"{final_te_05.get('dice', 0):.4f}"
        )

        test_row = cfg.copy()
        test_row["seed"] = SEED
        test_row["split"] = "Test"
        test_row["eval_setting"] = "Dice@0.5"
        test_row["thr_used"] = 0.5
        test_row.update(final_te_05)

        if "loss" in test_row:
            del test_row["loss"]

        test_row["best_val_dice@0.5"] = best_val_dice_05

        test_results_df.append(test_row)

    # =================================================================
    # Release resources before the next experiment
    # =================================================================
    del model, optimizer, scheduler, scaler, loss_fn
    del dl_tr, dl_va, dl_te

    torch.cuda.empty_cache()
    gc.collect()


# =====================================================================
# Compile and Save Main Train/Validation Results
# =====================================================================
print("\n" + "="*60)
print("🎯 FINAL TRAINING AND VALIDATION RESULTS:")
print("="*60)

df = pd.DataFrame(results_df)

# Exclude these metrics from the displayed and saved tables.
# Their calculations remain unchanged.
excluded_metrics = [
    "hd95",
    "recall_at_95_spec",
    "prevalence"
]

df = df.drop(columns=excluded_metrics, errors="ignore")

cols = list(df.columns)

preferred_order = [
    "name", "use_25d", "enc_att", "dec_att", "use_crop", "seed", "split",
    "eval_setting", "thr_used",

    # Main metrics
    "dice", "iou", "precision", "recall", "accuracy", "specificity",

    # Discrimination metrics
    "auc", "average_precision",

    # Confusion matrix
    "TP", "TN", "FP", "FN",

    # Training bookkeeping
    "best_val_dice@0.5"
]

final_cols = [c for c in preferred_order if c in cols]
final_cols += [c for c in cols if c not in preferred_order]

df = df[final_cols]

csv_path = (
    f"Final_Ablation_Results_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
)

df.to_csv(csv_path, index=False)

display(df)

print(f"💾 Train/Validation results saved: {csv_path}")


# =====================================================================
# Compile and Save Separate Test Results
# =====================================================================
print("\n" + "="*60)
print("🧪 FINAL TEST RESULTS (BASELINE AND DECATT):")
print("="*60)

if test_results_df:

    test_df = pd.DataFrame(test_results_df)

    # Exclude the same three metrics from the Test table.
    test_df = test_df.drop(
        columns=excluded_metrics,
        errors="ignore"
    )

    test_preferred_order = [
        "name", "use_25d", "enc_att", "dec_att", "use_crop",
        "seed", "split", "eval_setting", "thr_used",

        # Main metrics
        "dice", "iou", "precision", "recall", "accuracy", "specificity",

        # Discrimination metrics
        "auc", "average_precision",

        # Confusion matrix
        "TP", "TN", "FP", "FN",

        # Training bookkeeping
        "best_val_dice@0.5"
    ]

    test_cols = list(test_df.columns)

    final_test_cols = [
        c for c in test_preferred_order if c in test_cols
    ]

    final_test_cols += [
        c for c in test_cols if c not in test_preferred_order
    ]

    test_df = test_df[final_test_cols]

    test_csv_path = (
        f"Final_Test_Results_Baseline_DecAtt_"
        f"patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.csv"
    )

    test_df.to_csv(test_csv_path, index=False)

    display(test_df)

    print(f"💾 Test results saved: {test_csv_path}")

else:
    print("No Test results were generated.")


# =====================================================================
# Save Training Logs
# =====================================================================
if train_logs:

    train_log_df = pd.DataFrame(train_logs)

    log_path = (
        f"Training_Logs_patch{PATCH_SIZE}_{IMG_FORMAT}_S{SEED}.xlsx"
    )

    train_log_df.to_excel(log_path, index=False)

    print(f"🧾 Training logs saved: {log_path}")


🚀 Starting Ablation Study with 5 experiments...

🏆 Run 1/5 : 2D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Train Loss: 0.2839 | Val Loss: 0.2391 | Val Dice@0.5: 0.6011 | Best@0.5: 0.6011
   Ep 005/100 | LR: 3.00e-04 | Train Loss: 0.1165 | Val Loss: 0.2356 | Val Dice@0.5: 0.6398 | Best@0.5: 0.6837
   Ep 010/100 | LR: 3.00e-04 | Train Loss: 0.0911 | Val Loss: 0.2134 | Val Dice@0.5: 0.6744 | Best@0.5: 0.7041
   Ep 015/100 | LR: 1.50e-04 | Train Loss: 0.0761 | Val Loss: 0.2021 | Val Dice@0.5: 0.6768 | Best@0.5: 0.7041
   🛑 Early stopping triggered! No improvement for 10 epochs.
✅ Training 2D_Baseline finished in 49.7 min.
⏳ Calculating final Train/Validation metrics (@0.5 only)...
   Final Dice Train/Val @0.5: 0.7965 / 0.7041

🧪 Evaluating 2D_Baseline on the Test set...
   Test Dice@0.5: 0.6875

🏆 Run 2/5 : 2.5D_Baseline
Training samples: 9,845
Validation samples: 2,082
Test samples: 2,016
   Ep 001/100 | LR: 3.00e-04 | Tra

,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,45,Train,Dice@0.5,0.5,0.796508,...,0.852845,0.999425,0.999619,0.991326,0.790734,1133005,1005904074,383426,195495,0.704077
1,2D_Baseline,False,False,False,False,45,Val,Dice@0.5,0.5,0.704077,...,0.664379,0.999205,0.999682,0.960024,0.690367,201621,212825697,67630,101852,0.704077
2,2.5D_Baseline,True,False,False,False,45,Train,Dice@0.5,0.5,0.818612,...,0.903007,0.999486,0.999610,0.994396,0.816335,1169800,1005927792,392758,125650,0.712307
3,2.5D_Baseline,True,False,False,False,45,Val,Dice@0.5,0.5,0.712307,...,0.674149,0.999225,0.999688,0.929997,0.669136,204586,212826954,66373,98887,0.712307
4,2.5D_DecAtt,True,False,True,False,45,Train,Dice@0.5,0.5,0.840415,...,0.869394,0.999570,0.999740,0.993052,0.849419,1141528,1006040945,262039,171488,0.735210
5,2.5D_DecAtt,True,False,True,False,45,Val,Dice@0.5,0.5,0.735210,...,0.672719,0.999310,0.999776,0.911261,0.676098,204152,212845595,47732,99321,0.735210
6,2.5D_EncAtt,True,True,False,False,45,Train,Dice@0.5,0.5,0.842368,...,0.917743,0.999558,0.999664,0.995543,0.856090,1189050,1005981936,338440,106574,0.730945
7,2.5D_EncAtt,True,True,False,False,45,Val,Dice@0.5,0.5,0.730945,...,0.694774,0.999272,0.999706,0.947905,0.713549,210845,212830734,62593,92628,0.730945
8,2.5D_FullAtt,True,True,True,False,45,Train,Dice@0.5,0.5,0.846792,...,0.905587,0.999575,0.999697,0.994367,0.841924,1183657,1006004031,304908,123404,0.777676
9,2.5D_FullAtt,True,True,True,False,45,Val,Dice@0.5,0.5,0.777676,...,0.765706,0.999377,0.999710,0.949996,0.741475,232371,212831567,61760,71102,0.777676


💾 Train/Validation results saved: Final_Ablation_Results_patch320_PNG_S45.csv

🧪 FINAL TEST RESULTS (BASELINE AND DECATT):


,name,use_25d,enc_att,dec_att,use_crop,seed,split,eval_setting,thr_used,dice,...,recall,accuracy,specificity,auc,average_precision,TP,TN,FP,FN,best_val_dice@0.5
0,2D_Baseline,False,False,False,False,45,Test,Dice@0.5,0.5,0.687499,...,0.664190,0.999238,0.999661,0.973313,0.695024,172965,206108193,69792,87450,0.704077
1,2.5D_DecAtt,True,False,True,False,45,Test,Dice@0.5,0.5,0.696348,...,0.606701,0.999333,0.999828,0.946135,0.688883,157994,206142615,35370,102421,0.735210


💾 Test results saved: Final_Test_Results_Baseline_DecAtt_patch320_PNG_S45.csv
🧾 Training logs saved: Training_Logs_patch320_PNG_S45.xlsx
